#### document datastructure


In [5]:
from langchain_core.documents import Document

In [6]:
from typing import Any,List

In [7]:

document_one : Document = Document(
    page_content="Hello, world!", metadata={
        "source": "https://example.com",
        "date_created":"2020-04-98",
        "page":3,
        "author":"Nimesh"

        }
)

In [8]:
print(document_one)

page_content='Hello, world!' metadata={'source': 'https://example.com', 'date_created': '2020-04-98', 'page': 3, 'author': 'Nimesh'}


In [9]:
import os

os.makedirs("../data/textfiles",exist_ok=True)

In [10]:
sample_text = {"../data/textfiles/text_one.txt":"""RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval with text generation. 
Instead of relying only on a model's internal memory, RAG first searches a document collection for relevant context. 
The retrieved passages are then provided to the language model to produce a more accurate and grounded response.

A typical RAG pipeline has five steps:
1. Data ingestion: collect documents from sources like PDFs, web pages, and notes.
2. Chunking: split large documents into smaller text chunks.
3. Embedding: convert each chunk into a vector representation.
4. Indexing: store vectors in a vector database for fast similarity search.
5. Retrieval + generation: find top matching chunks for a user query and generate an answer using those chunks.

RAG is useful for question answering, internal knowledge assistants, customer support bots, and document search systems. 
Its main benefits are reduced hallucinations, better factual accuracy, and easy updates by changing the document store.

Common challenges include poor chunk size, weak embeddings, and missing metadata filters. 
Good practices include cleaning text, storing source metadata, and evaluating retrieval quality with real user questions.
"""}

In [11]:
for filepath,content in sample_text.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("file created")

file created


In [12]:
from langchain_community.document_loaders import TextLoader

text_loader = TextLoader(file_path="../data/textfiles/text_one",encoding="utf-8")

document = text_loader.load()
print(document)


[Document(metadata={'source': '../data/textfiles/text_one'}, page_content="RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval with text generation. \nInstead of relying only on a model's internal memory, RAG first searches a document collection for relevant context. \nThe retrieved passages are then provided to the language model to produce a more accurate and grounded response.\n\nA typical RAG pipeline has five steps:\n1. Data ingestion: collect documents from sources like PDFs, web pages, and notes.\n2. Chunking: split large documents into smaller text chunks.\n3. Embedding: convert each chunk into a vector representation.\n4. Indexing: store vectors in a vector database for fast similarity search.\n5. Retrieval + generation: find top matching chunks for a user query and generate an answer using those chunks.\n\nRAG is useful for question answering, internal knowledge assistants, customer support bots, and document search systems. \nIts main bene

In [13]:
from langchain_community.document_loaders import DirectoryLoader,PyPDFLoader

dir_loader = DirectoryLoader(
    path="../data/pdfs",
    glob="*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)

print(dir_loader.load())

100%|██████████| 3/3 [00:00<00:00,  9.57it/s]

[Document(metadata={'producer': 'Skia/PDF m130', 'creator': 'Chromium', 'creationdate': '2026-03-04T08:46:39+00:00', 'title': 'Anschreiben für Bewerbung Nimesh.pdf', 'moddate': '2026-03-04T08:46:39+00:00', 'source': '..\\data\\pdfs\\Anschreiben_für_Bewerbung_Nimesh.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Essen, 03. M ärz 2026\nInitiativbewerbung als Software-Entwickler\nSehr geehrte Damen und Herren,\nauf der Suche nach einer neuen beru\x00ichen Herausforderung bin ich auf die\nInternetpräsenz Ihres Unternehmens aufmerksam geworden und möchte Ihnen hiermit\nmeine Zusammenarbeit anbieten.\nZuletzt habe ich als Softwareentwickler bei der Werkbank GmbH, einer Digitalagentur aus\nBochum, gearbeitet. Dort programmierte ich in Python und war für die Weiterentwicklung\nvon Websites zuständig. Unter anderem habe ich mit dem Wagtail CM S gearbeitet und viel\nBackend-Entwicklung mit den Frameworks Django, Django REST und HTM X gemacht.\nObwohl ich nur sieben M onate 

In [14]:
from pathlib import Path

In [15]:

def load_documents(data_dir: Path) -> List[Document]:
    documents: List[Document] = []
    len_total_files = len([item for item in data_dir.iterdir()])
    print(f"found {len_total_files} files/objects to process")

    for file_path in data_dir.iterdir():

        if not file_path.is_file():
            print(
                f"{file_path} : is not a file  ! Processing Aborting ..\n",
            )
            continue
        file_ext = ((str(file_path).split("."))[-1]).lower()
        if file_ext not in ["txt", "pdf"]:
            print(f"Wrong file format {file_path} ! Processing Aborting ..\n")
            continue
        print(f"processing: {file_path.name} ")
        try:
            if file_ext == "pdf":
                doctype = PyPDFLoader(file_path=file_path)
                loader = doctype.load()
            elif file_ext == "txt":
                doctype = TextLoader(file_path=file_path, encoding="utf-8")
                loader = doctype.load()
            else:
                continue
        except Exception as e:
            print("Error occured while loading document from !!", file_path, e)
            continue
        for docs in loader:
            docs.metadata.update(
                {
                    "source": str(file_path),
                    "filename": file_path.name,
                    "extension": file_ext,
                }
            )
        documents.extend(loader)
        print(f" ✓ loaded {len(loader)} pages.. \n\n")
    print(f"Total  documents processed ✓ : {len(documents)} ")
    return documents


In [16]:
data_dir : str = Path("../data")
documents=load_documents(data_dir)

found 5 files/objects to process
processing: Anschreiben_für_Bewerbung_Nimesh.pdf 
 ✓ loaded 2 pages.. 


processing: dsfsdfas.txt 
 ✓ loaded 1 pages.. 


..\data\pdfs : is not a file  ! Processing Aborting ..

..\data\textfiles : is not a file  ! Processing Aborting ..

..\data\vector_store : is not a file  ! Processing Aborting ..

Total  documents processed ✓ : 3 


### Text splitting into chunks

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_chunk(
    document: List[Document], chunk_size: int = 2000, chunk_overlap: int = 200
):
    total_chunks = []
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )

    chunks = text_splitter.split_documents(documents=document)
    total_chunks.extend(chunks)
    print(f"Splitted {len(document)} Documents into {len(total_chunks)} chunks ✓ \n")
    return total_chunks



In [18]:
documents = split_chunk(documents)
print(documents)


Splitted 3 Documents into 14 chunks ✓ 

[Document(metadata={'producer': 'Skia/PDF m130', 'creator': 'Chromium', 'creationdate': '2026-03-04T08:46:39+00:00', 'title': 'Anschreiben für Bewerbung Nimesh.pdf', 'moddate': '2026-03-04T08:46:39+00:00', 'source': '..\\data\\Anschreiben_für_Bewerbung_Nimesh.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'filename': 'Anschreiben_für_Bewerbung_Nimesh.pdf', 'extension': 'pdf'}, page_content='Essen, 03. M ärz 2026\nInitiativbewerbung als Software-Entwickler\nSehr geehrte Damen und Herren,\nauf der Suche nach einer neuen beru\x00ichen Herausforderung bin ich auf die\nInternetpräsenz Ihres Unternehmens aufmerksam geworden und möchte Ihnen hiermit\nmeine Zusammenarbeit anbieten.\nZuletzt habe ich als Softwareentwickler bei der Werkbank GmbH, einer Digitalagentur aus\nBochum, gearbeitet. Dort programmierte ich in Python und war für die Weiterentwicklung\nvon Websites zuständig. Unter anderem habe ich mit dem Wagtail CM S gearbeitet und viel\nBac

In [19]:
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List


class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model {self.model_name} loaded successfully!")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {str(e)}")
            raise

    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not found")
        print(f"Generating embeddings for {len(texts)} text(s)...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"✓ Generated embeddings with shape: {embeddings.shape}")
        return embeddings


embedding_manager = EmbeddingManager()
# embedding_manager.generate_embedding()

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4085.97it/s]


Model all-MiniLM-L6-v2 loaded successfully!


In [20]:
def create_embedding_list(documents):
    embeddings = []
    for doc in documents:
        vectors = embedding_manager.generate_embedding(doc.page_content)
        embeddings.append(vectors)
    return embeddings
    

In [21]:
embeddings = create_embedding_list(documents)

Generating embeddings for 1941 text(s)...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.89it/s]


✓ Generated embeddings with shape: (384,)
Generating embeddings for 477 text(s)...


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.27it/s]


✓ Generated embeddings with shape: (384,)
Generating embeddings for 297 text(s)...


...

✓ Generated embeddings with shape: (384,)
Generating embeddings for 1910 text(s)...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.86it/s]


✓ Generated embeddings with shape: (384,)
Generating embeddings for 212 text(s)...


Batches: 100%|██████████| 1/1 [00:00<00:00, 51.65it/s]

✓ Generated embeddings with shape: (384,)


In [ ]:
import chromadb
import os
import hashlib
from langchain_core.documents import Document
from typing import List
import numpy as np


class VectorStore:
    def __init__(
        self,
        collection_name: str = "pdf-documents",
        persist_directory: str = "../data/vector_store",
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embedding for RAG"},
            )
            print(f"Vector Store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error occured {e}")
            raise

    def create_chunk_id(self, chunk,index):
        source = chunk.metadata.get("source", "")
        page = chunk.metadata.get("page", "")
        text = chunk.page_content
        raw = f"{source}|{page}|{text}|{index}".encode("utf-8")
        return hashlib.sha256(raw).hexdigest()

    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise RuntimeError("Something went wrong")
        print(f"Adding {len(documents)} vector")

        # prepare data for chroma db
        ids = []
        documents_text = []
        embeddings_list = []
        metadatas = []

        for  index,(chunk, embedding) in enumerate(zip(documents, embeddings)):
            ids.append(self.create_chunk_id(chunk,index))
            documents_text.append(chunk.page_content)
            embeddings_list.append(embedding.tolist())
            metadata = dict(chunk.metadata)
            metadata["content_length"] = len(chunk.page_content)
            metadatas.append(metadata)

        self.collection.upsert(
            ids=ids,
            documents=documents_text,
            embeddings=embeddings_list,
            metadatas=metadatas,
        )

        print(f"[INFO] Stored {len(documents)} chunks in ChromaDB")


vectorstore = VectorStore()


Vector Store initialized. Collection: pdf-documents
Existing documents in collection: 0


In [23]:
print(len(documents),len(embeddings))

14 14


In [25]:
vectorstore.add_documents(documents,embeddings)

Adding 14 vector


DuplicateIDError: Expected IDs to be unique, found duplicates of: 0512f685a05879d5b1c67cfd2bdb08844873d30ee1971908dc442a5bad869419, 8432ed80023a52300b0619c37ea5396c2cc717c6afcae78a6a4baaed25ca21dc in upsert.